In [1]:
import sys, json, logging
from pathlib import Path
import yaml
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

PROJECT_ROOT = Path("/home/yli94/CLIF/OHCA-RL")
CODE_DIR     = PROJECT_ROOT / "code"
CONFIG_DIR   = PROJECT_ROOT / "config"
OUT_DIR      = PROJECT_ROOT / "output" / "intermediate"
MODEL_DIR    = PROJECT_ROOT / "output" / "model"
FINAL_DIR    = PROJECT_ROOT / "output" / "final"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
FINAL_DIR.mkdir(parents=True, exist_ok=True)

# Reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# Configs (carried for any reward/RL hyperparams stored in YAML)
def deep_merge(base, override):
    out = dict(base) if base else {}
    for k, v in (override or {}).items():
        out[k] = deep_merge(out[k], v) if (k in out and isinstance(out[k], dict)
                                            and isinstance(v, dict)) else v
    return out

with open(CONFIG_DIR / "ohca_rl_config.yaml") as f:
    _base = yaml.safe_load(f)
_local_path = CONFIG_DIR / "ohca_rl_config_local.yaml"
_local = (yaml.safe_load(open(_local_path)) or {}) if _local_path.exists() else {}
ohca_config = deep_merge(_base, _local)

logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger("05_training")

# Force CPU (per user request)
device = torch.device("cpu")
print(f"Torch device: {device}")
print(f"PyTorch version: {torch.__version__}")

# ── Load training data ──
df = pd.read_parquet(OUT_DIR / "bucketed_with_reward.parquet")
df["hospitalization_id"] = df["hospitalization_id"].astype(str)

print(f"\nLoaded bucketed_with_reward.parquet:")
print(f"  Rows     : {len(df):,}")
print(f"  Cols     : {df.shape[1]}")
print(f"  Patients : {df['hospitalization_id'].nunique():,}")

# Quick verification — required columns
_required = ["hospitalization_id", "hour", "action_tier", "reward", "return",
             "mask_off", "mask_low", "mask_med", "mask_high", "mask_vhigh"]
_missing = [c for c in _required if c not in df.columns]
if _missing:
    print(f"\n⚠️  MISSING: {_missing}")
else:
    print(f"\n✓ All required columns present")

# Quick stats
print(f"\nAction tier distribution (training data):")
for tier, n in df["action_tier"].value_counts().sort_index().items():
    print(f"  Tier {tier}: {n:>6,}  ({n/len(df)*100:5.1f}%)")

print(f"\nReward distribution:")
print(df["reward"].describe(percentiles=[.05, .5, .95]).round(2).to_string())

Torch device: cpu
PyTorch version: 2.10.0+cu128

Loaded bucketed_with_reward.parquet:
  Rows     : 27,700
  Cols     : 101
  Patients : 700

✓ All required columns present

Action tier distribution (training data):
  Tier 0:  4,115  ( 14.9%)
  Tier 1:  4,969  ( 17.9%)
  Tier 2:  8,931  ( 32.2%)
  Tier 3:  5,343  ( 19.3%)
  Tier 4:  4,342  ( 15.7%)

Reward distribution:
count    27700.00
mean        -1.06
std         14.26
min       -100.63
5%          -0.36
50%          0.33
95%          0.39
max        100.63


In [2]:
# ── Define which columns are state features ──
# Drop: IDs, metadata, actions, masks, rewards, returns, categoricals, leakage
_drop_id_meta = [
    "hospitalization_id", "hour",
    "anchor_dttm", "anchor_source",
    "exit_hour", "is_scaffold", "window_close_reason",
    "cpc_tier", "cpc_tier_x", "cpc_tier_y", "survival_status",
]
_drop_action = ["action_tier", "action_label", "med_cont_nee"]
_drop_mask   = ["mask_off", "mask_low", "mask_med", "mask_high", "mask_vhigh"]
_drop_reward = ["r_intermediate", "r_terminal", "reward", "return",
                "raw_intermediate", "raw_intermediate_unit"]   # ← LEAKAGE
_drop_categorical = [c for c in df.columns
                     if c.startswith(("resp_device_", "resp_mode_", "resp_vent_brand_",
                                      "resp_tracheostomy", "adt_"))]
_drop_extra = ["in_decision_window", "first_vaso_hour"]

drop_cols = set(_drop_id_meta + _drop_action + _drop_mask + _drop_reward +
                _drop_categorical + _drop_extra)
state_cols = [c for c in df.columns if c not in drop_cols]

_non_numeric = [c for c in state_cols if not pd.api.types.is_numeric_dtype(df[c])]
if _non_numeric:
    print(f"⚠️  Non-numeric state cols (will be dropped): {_non_numeric}")
    state_cols = [c for c in state_cols if c not in _non_numeric]

print(f"State features ({len(state_cols)} total):")
for i, c in enumerate(state_cols):
    print(f"  {i+1:>2}. {c}")

# Sanity check: confirm no reward-related columns slipped through
_leaks = [c for c in state_cols if "reward" in c or "return" in c or "raw_inter" in c]
if _leaks:
    print(f"\n⚠️  LEAKAGE detected: {_leaks}")
else:
    print(f"\n✓ No reward leakage in state features")

# ── Patient-level train/val split (80/20) ──
unique_pts = df["hospitalization_id"].unique()
np.random.shuffle(unique_pts)

n_train = int(0.8 * len(unique_pts))
train_pts = set(unique_pts[:n_train])
val_pts = set(unique_pts[n_train:])

df_train = df[df["hospitalization_id"].isin(train_pts)].copy()
df_val   = df[df["hospitalization_id"].isin(val_pts)].copy()

print(f"\nPatient-level split (seed={SEED}, 80/20):")
print(f"  Train: {len(train_pts):,} patients, {len(df_train):,} decision points")
print(f"  Val  : {len(val_pts):,} patients, {len(df_val):,} decision points")

print(f"\nAction tier balance — train vs val:")
for tier in range(5):
    _t_pct = (df_train["action_tier"] == tier).mean() * 100
    _v_pct = (df_val["action_tier"] == tier).mean() * 100
    print(f"  Tier {tier}: train {_t_pct:5.1f}%, val {_v_pct:5.1f}%")

patient_disp = pd.read_parquet(OUT_DIR / "patient_disposition.parquet")
patient_disp["hospitalization_id"] = patient_disp["hospitalization_id"].astype(str)
_disp_train = patient_disp[patient_disp["hospitalization_id"].isin(train_pts)]
_disp_val   = patient_disp[patient_disp["hospitalization_id"].isin(val_pts)]

print(f"\nCPC tier balance — train vs val:")
for tier in ["CPC1_2", "CPC3", "CPC4", "CPC5"]:
    _t_n = (_disp_train["cpc_tier"] == tier).sum()
    _v_n = (_disp_val["cpc_tier"] == tier).sum()
    _t_pct = _t_n / len(_disp_train) * 100
    _v_pct = _v_n / len(_disp_val) * 100
    print(f"  {tier:6s}: train {_t_n:>3} ({_t_pct:4.1f}%),  val {_v_n:>3} ({_v_pct:4.1f}%)")

State features (66 total):
   1. vital_dbp
   2. vital_heart_rate
   3. vital_height_cm
   4. vital_map
   5. vital_respiratory_rate
   6. vital_sbp
   7. vital_spo2
   8. vital_temp_c
   9. vital_weight_kg
  10. lab_bicarbonate
  11. lab_bun
  12. lab_calcium_total
  13. lab_chloride
  14. lab_creatinine
  15. lab_glucose_serum
  16. lab_hemoglobin
  17. lab_lactate
  18. lab_magnesium
  19. lab_pco2_arterial
  20. lab_ph_arterial
  21. lab_po2_arterial
  22. lab_potassium
  23. lab_so2_arterial
  24. lab_sodium
  25. resp_fio2_set
  26. resp_lpm_set
  27. resp_peep_set
  28. resp_resp_rate_obs
  29. resp_resp_rate_set
  30. resp_tidal_volume_obs
  31. resp_tidal_volume_set
  32. assess_gcs_total
  33. assess_rass
  34. med_cont_angiotensin
  35. med_cont_cisatracurium
  36. med_cont_clevidipine
  37. med_cont_dexmedetomidine
  38. med_cont_dobutamine
  39. med_cont_dopamine
  40. med_cont_epinephrine
  41. med_cont_ketamine
  42. med_cont_midazolam
  43. med_cont_milrinone
  44. med_

In [3]:
mask_cols = ["mask_off", "mask_low", "mask_med", "mask_high", "mask_vhigh"]

def build_transitions(df_subset, state_cols, mask_cols):
    """Build (s, a, r, s', done, mask') tuples within each patient."""
    df_subset = df_subset.sort_values(["hospitalization_id", "hour"]).reset_index(drop=True)
    
    # done = 1 if this is the LAST row for the patient (not just NaN in next_state)
    # Use cumcount(ascending=False) — counts from end of group, so 0 = last row
    grouped = df_subset.groupby("hospitalization_id", sort=False)
    done = (grouped.cumcount(ascending=False) == 0).astype(int)
    
    # Next state and next mask (using shift -1 within group)
    next_state = grouped[state_cols].shift(-1)
    next_mask  = grouped[mask_cols].shift(-1)
    
    # Fill terminal next_state with 0 (won't be used since done=1)
    next_state = next_state.fillna(0)
    next_mask = next_mask.fillna(1)
    
    return {
        "state":      df_subset[state_cols].values.astype(np.float32),
        "action":     df_subset["action_tier"].values.astype(np.int64),
        "reward":     df_subset["reward"].values.astype(np.float32),
        "next_state": next_state.values.astype(np.float32),
        "next_mask":  next_mask.values.astype(np.float32),
        "done":       done.values.astype(np.float32),
        "patient_id": df_subset["hospitalization_id"].values,
        "hour":       df_subset["hour"].values,
    }

print("Building transition tuples (with corrected done definition)...")
train_data = build_transitions(df_train, state_cols, mask_cols)
val_data   = build_transitions(df_val,   state_cols, mask_cols)

print(f"\nTrain transitions: {len(train_data['state']):,}")
print(f"  Terminal      : {int(train_data['done'].sum()):,}")
print(f"  Non-terminal  : {int((1 - train_data['done']).sum()):,}")

print(f"\nVal transitions  : {len(val_data['state']):,}")
print(f"  Terminal      : {int(val_data['done'].sum()):,}")
print(f"  Non-terminal  : {int((1 - val_data['done']).sum()):,}")

# Sanity: terminals = patients
n_train_pts = len(set(train_data['patient_id']))
n_val_pts   = len(set(val_data['patient_id']))
print(f"\nSanity (terminals = patients):")
print(f"  Train: {int(train_data['done'].sum())} terminals = {n_train_pts} patients  "
      f"({'✓' if train_data['done'].sum() == n_train_pts else '✗'})")
print(f"  Val  : {int(val_data['done'].sum())} terminals = {n_val_pts} patients  "
      f"({'✓' if val_data['done'].sum() == n_val_pts else '✗'})")

# Sanity: the reward at terminal should be one of the terminal values (±100 / ±40)
_terminal_rewards = train_data['reward'][train_data['done'] == 1]
print(f"\nTerminal reward magnitudes (train):")
print(f"  Around +100: {((_terminal_rewards > 80) & (_terminal_rewards < 120)).sum()}")
print(f"  Around +40 : {((_terminal_rewards > 20) & (_terminal_rewards < 60)).sum()}")
print(f"  Around -40 : {((_terminal_rewards < -20) & (_terminal_rewards > -60)).sum()}")
print(f"  Around -100: {((_terminal_rewards < -80) & (_terminal_rewards > -120)).sum()}")

Building transition tuples (with corrected done definition)...

Train transitions: 21,602
  Terminal      : 560
  Non-terminal  : 21,042

Val transitions  : 6,098
  Terminal      : 140
  Non-terminal  : 5,958

Sanity (terminals = patients):
  Train: 560 terminals = 560 patients  (✓)
  Val  : 140 terminals = 140 patients  (✓)

Terminal reward magnitudes (train):
  Around +100: 81
  Around +40 : 43
  Around -40 : 92
  Around -100: 344


In [4]:
# ── Compute imputation values (medians) from training data ──
_train_state_df = pd.DataFrame(train_data["state"], columns=state_cols)
_val_state_df   = pd.DataFrame(val_data["state"],   columns=state_cols)
_train_next_df  = pd.DataFrame(train_data["next_state"], columns=state_cols)
_val_next_df    = pd.DataFrame(val_data["next_state"],   columns=state_cols)

# Per-feature median from TRAINING data only (no val leakage)
feature_medians = _train_state_df.median(axis=0)
# Replace any NaN medians (constant-NaN features) with 0
feature_medians = feature_medians.fillna(0)

print("Imputation strategy: median per feature from training data")
print(f"Top 5 features by NaN count being imputed (train state):")
_top_nan = _train_state_df.isna().sum().sort_values(ascending=False).head(5)
for c, n in _top_nan.items():
    print(f"  {c:40s}  {n:>6,} NaN → impute with median {feature_medians[c]:.2f}")

# Apply imputation to all 4 state matrices
_train_state_df = _train_state_df.fillna(feature_medians)
_val_state_df   = _val_state_df.fillna(feature_medians)
_train_next_df  = _train_next_df.fillna(feature_medians)
_val_next_df    = _val_next_df.fillna(feature_medians)

# Confirm no NaN left
print(f"\nPost-imputation NaN check:")
print(f"  Train state: {_train_state_df.isna().sum().sum()}")
print(f"  Val state  : {_val_state_df.isna().sum().sum()}")
print(f"  Train next : {_train_next_df.isna().sum().sum()}")
print(f"  Val next   : {_val_next_df.isna().sum().sum()}")

# ── Z-score normalization (stats from training data) ──
feature_mean = _train_state_df.mean(axis=0)
feature_std  = _train_state_df.std(axis=0)
# Avoid divide-by-zero for constant features
feature_std = feature_std.replace(0, 1.0)

print(f"\nZ-score normalization stats (from training):")
print(f"  Mean range: [{feature_mean.min():.2f}, {feature_mean.max():.2f}]")
print(f"  Std range : [{feature_std.min():.2f}, {feature_std.max():.2f}]")
print(f"  Features with std=0 (replaced with 1): "
      f"{(_train_state_df.std(axis=0) == 0).sum()}")

def normalize(df_state):
    return ((df_state - feature_mean) / feature_std).values.astype(np.float32)

train_data["state"]      = normalize(_train_state_df)
train_data["next_state"] = normalize(_train_next_df)
val_data["state"]        = normalize(_val_state_df)
val_data["next_state"]   = normalize(_val_next_df)

# Sanity: post-normalization
print(f"\nPost-normalization stats (train state):")
print(f"  Min : {train_data['state'].min():.2f}")
print(f"  Max : {train_data['state'].max():.2f}")
print(f"  Mean: {train_data['state'].mean():.2f}")
print(f"  Std : {train_data['state'].std():.2f}")

# Save the normalization stats for downstream use (eval, deployment)
norm_stats = pd.DataFrame({
    "feature": state_cols,
    "median":  feature_medians.values,
    "mean":    feature_mean.values,
    "std":     feature_std.values,
})
norm_stats.to_parquet(OUT_DIR / "normalization_stats.parquet", index=False)
print(f"\nSaved normalization stats → {OUT_DIR / 'normalization_stats.parquet'}")
print(f"  (for use in evaluation, deployment, external validation)")

Imputation strategy: median per feature from training data
Top 5 features by NaN count being imputed (train state):
  resp_lpm_set                              16,921 NaN → impute with median 3.00
  resp_tidal_volume_set                      1,928 NaN → impute with median 420.00
  vital_height_cm                            1,869 NaN → impute with median 170.00
  resp_resp_rate_set                         1,720 NaN → impute with median 20.00
  resp_tidal_volume_obs                      1,428 NaN → impute with median 426.00

Post-imputation NaN check:
  Train state: 0
  Val state  : 0
  Train next : 0
  Val next   : 0

Z-score normalization stats (from training):
  Mean range: [-3.13, 435.19]
  Std range : [0.00, 115.10]
  Features with std=0 (replaced with 1): 0

Post-normalization stats (train state):
  Min : -22.39
  Max : 146.97
  Mean: 0.00
  Std : 1.00

Saved normalization stats → /home/yli94/CLIF/OHCA-RL/output/intermediate/normalization_stats.parquet
  (for use in evaluation, dep

In [5]:
# Clip post-normalization features to ±5 std to prevent gradient explosions
# from extreme charting outliers (e.g. NEE=10 mcg/kg/min misentries)
CLIP_STD = 5.0
train_data["state"]      = np.clip(train_data["state"],      -CLIP_STD, CLIP_STD)
train_data["next_state"] = np.clip(train_data["next_state"], -CLIP_STD, CLIP_STD)
val_data["state"]        = np.clip(val_data["state"],        -CLIP_STD, CLIP_STD)
val_data["next_state"]   = np.clip(val_data["next_state"],   -CLIP_STD, CLIP_STD)

print(f"After clipping to ±{CLIP_STD} std:")
print(f"  Train state: min={train_data['state'].min():.2f}, max={train_data['state'].max():.2f}")
print(f"  Val state  : min={val_data['state'].min():.2f}, max={val_data['state'].max():.2f}")

# How many cells were clipped?
_total = train_data["state"].size + val_data["state"].size
_clipped = (
    np.abs(train_data["state"]).round(5) == CLIP_STD).sum() + (
    np.abs(val_data["state"]).round(5) == CLIP_STD).sum()
print(f"  Cells clipped: {_clipped:,} / {_total:,} ({100*_clipped/_total:.3f}%)")

After clipping to ±5.0 std:
  Train state: min=-5.00, max=5.00
  Val state  : min=-5.00, max=5.00
  Cells clipped: 6,509 / 1,828,200 (0.356%)


In [6]:
class DuelingDQN(nn.Module):
    """
    Dueling DQN: Q(s, a) = V(s) + A(s, a) - mean_a A(s, ·)
    
    Architecture:
      Shared trunk: state_dim → 128 → 128 (ReLU)
      Value head:   128 → 1
      Advantage:    128 → n_actions
    """
    def __init__(self, state_dim, n_actions=5, hidden_dim=128):
        super().__init__()
        self.trunk = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
        )
        self.value_head     = nn.Linear(hidden_dim, 1)
        self.advantage_head = nn.Linear(hidden_dim, n_actions)
        
        # Init: small last-layer weights for stable early training
        nn.init.uniform_(self.value_head.weight, -3e-3, 3e-3)
        nn.init.uniform_(self.advantage_head.weight, -3e-3, 3e-3)
    
    def forward(self, state):
        """Returns Q-values, shape (batch, n_actions)."""
        h = self.trunk(state)
        v = self.value_head(h)           # (batch, 1)
        a = self.advantage_head(h)        # (batch, n_actions)
        # Subtract mean advantage to enforce identifiability
        q = v + (a - a.mean(dim=1, keepdim=True))
        return q
    
    def predict_action(self, state, mask):
        """Greedy action selection with masking. state: (batch, state_dim), mask: (batch, n_actions)."""
        with torch.no_grad():
            q = self.forward(state)
            # Set forbidden actions to -inf so argmax never picks them
            q_masked = q.masked_fill(mask == 0, float("-inf"))
            return q_masked.argmax(dim=1)


# ── Instantiate ──
STATE_DIM   = len(state_cols)
N_ACTIONS   = 5
HIDDEN_DIM  = 128

online_net = DuelingDQN(STATE_DIM, N_ACTIONS, HIDDEN_DIM).to(device)
target_net = DuelingDQN(STATE_DIM, N_ACTIONS, HIDDEN_DIM).to(device)
target_net.load_state_dict(online_net.state_dict())  # Init same as online
target_net.eval()  # Target net never trains directly

# Parameter count
_n_params = sum(p.numel() for p in online_net.parameters())
print(f"Dueling DQN architecture:")
print(f"  Input dim : {STATE_DIM} state features")
print(f"  Hidden    : {HIDDEN_DIM} × 2 layers (ReLU)")
print(f"  Output    : V(s) + A(s, {N_ACTIONS} actions)")
print(f"  Params    : {_n_params:,}")
print(f"  Device    : {device}")

# Sanity forward pass
_dummy_state = torch.zeros((4, STATE_DIM), dtype=torch.float32, device=device)
_dummy_mask  = torch.ones((4, N_ACTIONS), dtype=torch.float32, device=device)
with torch.no_grad():
    _q = online_net(_dummy_state)
    _act = online_net.predict_action(_dummy_state, _dummy_mask)
print(f"\nSanity forward pass:")
print(f"  Input shape : {tuple(_dummy_state.shape)}")
print(f"  Q-values    : {tuple(_q.shape)}  range [{_q.min():.3f}, {_q.max():.3f}]")
print(f"  Predicted   : {_act.tolist()}")

Dueling DQN architecture:
  Input dim : 66 state features
  Hidden    : 128 × 2 layers (ReLU)
  Output    : V(s) + A(s, 5 actions)
  Params    : 25,862
  Device    : cpu

Sanity forward pass:
  Input shape : (4, 66)
  Q-values    : (4, 5)  range [-0.035, 0.103]
  Predicted   : [1, 1, 1, 1]


In [9]:
# Convert numpy → tensors (must run before training loop)
def _to_tensors(d):
    return {
        "state":      torch.tensor(d["state"],      dtype=torch.float32, device=device),
        "action":     torch.tensor(d["action"],     dtype=torch.long,    device=device),
        "reward":     torch.tensor(d["reward"],     dtype=torch.float32, device=device),
        "next_state": torch.tensor(d["next_state"], dtype=torch.float32, device=device),
        "next_mask":  torch.tensor(d["next_mask"],  dtype=torch.float32, device=device),
        "done":       torch.tensor(d["done"],       dtype=torch.float32, device=device),
    }

train_tensors = _to_tensors(train_data)
val_tensors   = _to_tensors(val_data)

print(f"Train tensors: state={tuple(train_tensors['state'].shape)}, "
      f"reward={tuple(train_tensors['reward'].shape)}")
print(f"Val tensors  : state={tuple(val_tensors['state'].shape)}, "
      f"reward={tuple(val_tensors['reward'].shape)}")

Train tensors: state=(21602, 66), reward=(21602,)
Val tensors  : state=(6098, 66), reward=(6098,)


In [14]:
# ============================================================
# TRAINING — Dueling Double DQN with Huber loss + BC regularization
# ============================================================
LR              = 1e-3
BATCH_SIZE      = 256
N_EPOCHS        = 200
TARGET_UPDATE_EVERY = 100
GRAD_CLIP       = 10.0
GAMMA           = 0.99
LOG_EVERY       = 10
EARLY_STOP_PATIENCE = 30
BC_LAMBDA       = 0.5            # BC regularization weight
HUBER_DELTA     = 1.0            # Huber loss threshold

# Re-init networks
online_net = DuelingDQN(STATE_DIM, N_ACTIONS, HIDDEN_DIM).to(device)
target_net = DuelingDQN(STATE_DIM, N_ACTIONS, HIDDEN_DIM).to(device)
target_net.load_state_dict(online_net.state_dict())
target_net.eval()

optimizer = torch.optim.Adam(online_net.parameters(), lr=LR)

def compute_td_target(rewards, next_states, next_masks, dones, gamma):
    """Double DQN target with action masking on next state."""
    with torch.no_grad():
        q_next_online = online_net(next_states)
        q_next_online_masked = q_next_online.masked_fill(next_masks == 0, float("-inf"))
        action_star = q_next_online_masked.argmax(dim=1)
        q_next_target = target_net(next_states)
        q_next_target_at_star = q_next_target.gather(1, action_star.unsqueeze(1)).squeeze(1)
        target = rewards + gamma * q_next_target_at_star * (1 - dones)
    return target

def train_epoch(global_step):
    online_net.train()
    n = len(train_tensors["state"])
    indices = torch.randperm(n)
    total_td_loss = 0.0
    total_bc_loss = 0.0
    total_loss    = 0.0
    n_batches     = 0
    
    for start in range(0, n, BATCH_SIZE):
        batch_idx = indices[start:start + BATCH_SIZE]
        s   = train_tensors["state"][batch_idx]
        a   = train_tensors["action"][batch_idx]
        r   = train_tensors["reward"][batch_idx]
        sp  = train_tensors["next_state"][batch_idx]
        mp  = train_tensors["next_mask"][batch_idx]
        d   = train_tensors["done"][batch_idx]
        
        # Forward: Q(s, ·) for all actions
        q_all = online_net(s)
        q_sa = q_all.gather(1, a.unsqueeze(1)).squeeze(1)
        
        # TD target (with action masking on next-state)
        target = compute_td_target(r, sp, mp, d, GAMMA)
        
        # Huber (smooth_l1) TD loss — robust to ±100 terminal magnitudes
        td_loss = F.smooth_l1_loss(q_sa, target, beta=HUBER_DELTA)
        
        # BC regularization: cross-entropy between softmax(Q) and observed clinician action
        # This pulls the policy toward clinician behavior, preventing failure modes like
        # "always VeryHigh" or "never Off". Standard for offline RL (Komorowski, BCQ, CQL-BC).
        bc_loss = F.cross_entropy(q_all, a)
        
        # Combined loss
        loss = td_loss + BC_LAMBDA * bc_loss
        
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(online_net.parameters(), GRAD_CLIP)
        optimizer.step()
        
        global_step += 1
        if global_step % TARGET_UPDATE_EVERY == 0:
            target_net.load_state_dict(online_net.state_dict())
        
        total_td_loss += td_loss.item()
        total_bc_loss += bc_loss.item()
        total_loss    += loss.item()
        n_batches     += 1
    
    return (total_loss / n_batches,
            total_td_loss / n_batches,
            total_bc_loss / n_batches,
            global_step)

def validate(tensors):
    """Validation: report combined Huber+BC loss + diagnostic action distribution."""
    online_net.eval()
    n = len(tensors["state"])
    total_loss = 0.0
    total_td_loss = 0.0
    total_bc_loss = 0.0
    n_batches = 0
    
    # Also track what actions the policy is selecting (for diagnostics)
    action_counts = np.zeros(5, dtype=int)
    
    with torch.no_grad():
        for start in range(0, n, BATCH_SIZE):
            end = min(start + BATCH_SIZE, n)
            s = tensors["state"][start:end]
            a = tensors["action"][start:end]
            r = tensors["reward"][start:end]
            sp = tensors["next_state"][start:end]
            mp = tensors["next_mask"][start:end]
            d = tensors["done"][start:end]
            
            q_all = online_net(s)
            q_sa = q_all.gather(1, a.unsqueeze(1)).squeeze(1)
            target = compute_td_target(r, sp, mp, d, GAMMA)
            
            td_loss = F.smooth_l1_loss(q_sa, target, beta=HUBER_DELTA)
            bc_loss = F.cross_entropy(q_all, a)
            loss = td_loss + BC_LAMBDA * bc_loss
            
            total_td_loss += td_loss.item()
            total_bc_loss += bc_loss.item()
            total_loss    += loss.item()
            n_batches     += 1
            
            # Policy action distribution (mask-respecting argmax)
            policy_a = q_all.argmax(dim=1).cpu().numpy()
            for ai in range(5):
                action_counts[ai] += (policy_a == ai).sum()
    
    return (total_loss / n_batches,
            total_td_loss / n_batches,
            total_bc_loss / n_batches,
            action_counts)

print(f"TRAINING — DDQN with Huber loss (δ={HUBER_DELTA}) + BC regularization (λ={BC_LAMBDA})")
print(f"  lr={LR}, batch={BATCH_SIZE}, target update every {TARGET_UPDATE_EVERY} steps")
print(f"  Note: BC loss pulls policy toward clinician behavior to prevent failure modes")
print("="*70)

history = {
    "train_loss": [], "train_td": [], "train_bc": [],
    "val_loss":   [], "val_td":   [], "val_bc":   [],
}
best_val_loss = float("inf")
best_epoch = 0
patience_counter = 0
global_step = 0

for epoch in range(1, N_EPOCHS + 1):
    train_loss, train_td, train_bc, global_step = train_epoch(global_step)
    val_loss, val_td, val_bc, action_counts = validate(val_tensors)
    
    history["train_loss"].append(train_loss); history["train_td"].append(train_td); history["train_bc"].append(train_bc)
    history["val_loss"].append(val_loss);     history["val_td"].append(val_td);     history["val_bc"].append(val_bc)
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch
        patience_counter = 0
        torch.save(online_net.state_dict(), MODEL_DIR / "ddqn_best.pt")
    else:
        patience_counter += 1
    
    if epoch % LOG_EVERY == 0 or epoch == 1:
        total = action_counts.sum()
        act_str = " ".join(f"{ai}={100*action_counts[ai]/total:.0f}%" for ai in range(5))
        print(f"Epoch {epoch:>3}/{N_EPOCHS}  "
              f"train: loss={train_loss:>7.2f} (td={train_td:>6.2f}, bc={train_bc:>5.3f})  "
              f"val: loss={val_loss:>7.2f}  best={best_val_loss:>7.2f} (ep {best_epoch})  "
              f"policy_dist: {act_str}")
    
    if patience_counter >= EARLY_STOP_PATIENCE:
        print(f"\nEarly stopping at epoch {epoch}")
        break

print(f"\nTraining complete — best val_loss={best_val_loss:.4f} at epoch {best_epoch}")
online_net.load_state_dict(torch.load(MODEL_DIR / "ddqn_best.pt"))
online_net.eval()

TRAINING — DDQN with Huber loss (δ=1.0) + BC regularization (λ=0.5)
  lr=0.001, batch=256, target update every 100 steps
  Note: BC loss pulls policy toward clinician behavior to prevent failure modes
Epoch   1/200  train: loss=   2.89 (td=  2.26, bc=1.260)  val: loss=   2.54  best=   2.54 (ep 1)  policy_dist: 0=10% 1=7% 2=50% 3=21% 4=12%
Epoch  10/200  train: loss=   2.56 (td=  2.47, bc=0.180)  val: loss=   2.68  best=   2.36 (ep 3)  policy_dist: 0=14% 1=15% 2=35% 3=21% 4=15%
Epoch  20/200  train: loss=   3.55 (td=  3.45, bc=0.191)  val: loss=   4.53  best=   2.36 (ep 3)  policy_dist: 0=14% 1=15% 2=34% 3=22% 4=15%
Epoch  30/200  train: loss=   8.03 (td=  7.88, bc=0.296)  val: loss=  10.29  best=   2.36 (ep 3)  policy_dist: 0=15% 1=15% 2=34% 3=19% 4=17%

Early stopping at epoch 33

Training complete — best val_loss=2.3641 at epoch 3


DuelingDQN(
  (trunk): Sequential(
    (0): Linear(in_features=66, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=128, bias=True)
    (3): ReLU()
  )
  (value_head): Linear(in_features=128, out_features=1, bias=True)
  (advantage_head): Linear(in_features=128, out_features=5, bias=True)
)

In [15]:
# ============================================================
# EVALUATION ON VAL SET
# ============================================================
import statsmodels.api as sm

print("="*70)
print("EVALUATION — concordance + adjusted outcome regression")
print("="*70)

# ── Policy actions on val ──
with torch.no_grad():
    val_q = online_net(val_tensors["state"])
    val_mask_now = torch.tensor(
        df_val.sort_values(["hospitalization_id", "hour"])[
            ["mask_off","mask_low","mask_med","mask_high","mask_vhigh"]
        ].values.astype(np.float32),
        device=device,
    )
    val_q_masked = val_q.masked_fill(val_mask_now == 0, float("-inf"))
    policy_action = val_q_masked.argmax(dim=1).cpu().numpy()

clinician_action = val_data["action"]
agreement = (policy_action == clinician_action).astype(int)
action_distance = np.abs(policy_action - clinician_action)

# ── Decision-point-level metrics ──
print(f"\nDecision-point concordance (val, n={len(policy_action):,}):")
print(f"  Exact agreement     : {agreement.mean()*100:.1f}%")
print(f"  Within 1 tier       : {(action_distance <= 1).mean()*100:.1f}%")
print(f"  Mean action distance: {action_distance.mean():.2f} tiers")

# ── Action distribution comparison ──
print(f"\nAction distribution — clinician vs policy:")
print(f"  {'Tier':<10} {'Clinician':>10} {'Policy':>10} {'Δ':>8}")
for tier in range(5):
    c_pct = (clinician_action == tier).mean() * 100
    p_pct = (policy_action == tier).mean() * 100
    label = ["Off", "Low", "Med", "High", "VHigh"][tier]
    print(f"  {tier} {label:<7} {c_pct:>9.1f}% {p_pct:>9.1f}% {p_pct - c_pct:>+7.1f}%")

# ── Confusion matrix ──
print(f"\nConfusion matrix (rows=clinician, cols=policy, % of decisions):")
cm_pct = np.zeros((5, 5))
for c in range(5):
    for p in range(5):
        cm_pct[c, p] = ((clinician_action == c) & (policy_action == p)).mean() * 100
print(" " * 10 + "".join(f"P:{t:<6}" for t in ["Off","Low","Med","High","VHi"]))
for c in range(5):
    label = ["Off","Low","Med","High","VHi"][c]
    row = f"  C:{label:<5} "
    for p in range(5):
        row += f"{cm_pct[c,p]:>7.1f}"
    print(row)

# ── Per-patient concordance ──
val_df_for_eval = pd.DataFrame({
    "hospitalization_id": val_data["patient_id"],
    "hour":               val_data["hour"],
    "clinician_action":   clinician_action,
    "policy_action":      policy_action,
    "agreement":          agreement,
    "action_distance":    action_distance,
})
per_patient = val_df_for_eval.groupby("hospitalization_id").agg(
    n_dp                 = ("agreement", "size"),
    mean_agreement       = ("agreement", "mean"),
    mean_action_distance = ("action_distance", "mean"),
).reset_index()

# ── Load confounders ──
patient_static = pd.read_parquet(OUT_DIR / "patient_static.parquet")
patient_static["hospitalization_id"] = patient_static["hospitalization_id"].astype(str)
patient_disp = pd.read_parquet(OUT_DIR / "patient_disposition.parquet")
patient_disp["hospitalization_id"] = patient_disp["hospitalization_id"].astype(str)
sofa_0_24 = pd.read_parquet(OUT_DIR / "sofa_0_24.parquet")
sofa_0_24["hospitalization_id"] = sofa_0_24["hospitalization_id"].astype(str)

per_patient = (per_patient
    .merge(patient_disp[["hospitalization_id", "cpc_tier"]],
           on="hospitalization_id", how="left")
    .merge(patient_static[["hospitalization_id", "age_at_admission", "sex_category"]],
           on="hospitalization_id", how="left")
    .merge(sofa_0_24[["hospitalization_id", "sofa_total_0_24"]],
           on="hospitalization_id", how="left"))

# Outcome encodings
per_patient["death"]        = (per_patient["cpc_tier"] == "CPC5").astype(int)
per_patient["good_outcome"] = (per_patient["cpc_tier"] == "CPC1_2").astype(int)
_cpc_order = {"CPC1_2": 0, "CPC3": 1, "CPC4": 2, "CPC5": 3}
per_patient["cpc_ordinal"] = per_patient["cpc_tier"].map(_cpc_order)
per_patient["sex_male"] = (per_patient["sex_category"].str.lower() == "male").astype(int)

# ── Concordance quartile + outcome table ──
per_patient["concordance_quartile"] = pd.qcut(
    per_patient["mean_agreement"], q=4,
    labels=["Q1 (lowest)", "Q2", "Q3", "Q4 (highest)"], duplicates="drop"
)
print(f"\nPer-patient concordance distribution (val, n={len(per_patient)}):")
print(per_patient["mean_agreement"].describe(percentiles=[.25, .5, .75]).round(3).to_string())

print(f"\nOutcome by concordance quartile:")
_outcome_table = (per_patient.groupby("concordance_quartile", observed=True)
                  .agg(n=("cpc_tier","count"),
                       mean_conc=("mean_agreement","mean"),
                       pct_CPC1_2=("cpc_tier", lambda s: (s == "CPC1_2").mean()*100),
                       pct_CPC3  =("cpc_tier", lambda s: (s == "CPC3").mean()*100),
                       pct_CPC4  =("cpc_tier", lambda s: (s == "CPC4").mean()*100),
                       pct_CPC5_death=("cpc_tier", lambda s: (s == "CPC5").mean()*100))
                  .round(1))
print(_outcome_table.to_string())

# ── Adjusted logistic regression ──
print(f"\n" + "="*70)
print("ADJUSTED OUTCOME REGRESSION (adjusting for age, sex, SOFA 0-24h)")
print("="*70)

_reg_df = per_patient.dropna(subset=["mean_agreement", "age_at_admission",
                                      "sex_male", "sofa_total_0_24"]).copy()

# Force everything to numeric (avoid object-dtype regression errors)
for col in ["mean_agreement", "age_at_admission", "sex_male", "sofa_total_0_24"]:
    _reg_df[col] = pd.to_numeric(_reg_df[col], errors="coerce").astype(float)
for col in ["death", "good_outcome"]:
    _reg_df[col] = _reg_df[col].astype(float)
_reg_df = _reg_df.dropna(subset=["mean_agreement", "age_at_admission",
                                  "sex_male", "sofa_total_0_24"]).copy()

_reg_df["agreement_10pct"] = _reg_df["mean_agreement"] * 10
print(f"\nRegression cohort: {len(_reg_df)} of {len(per_patient)} patients "
      f"(dropped {len(per_patient) - len(_reg_df)} with missing covariates)")
print(f"\nDtypes confirmed numeric:")
print(_reg_df[["agreement_10pct", "age_at_admission", "sex_male",
               "sofa_total_0_24", "death", "good_outcome"]].dtypes.to_string())

# ── Models ──
X_unadj = sm.add_constant(_reg_df[["agreement_10pct"]])
mod1 = sm.Logit(_reg_df["death"], X_unadj).fit(disp=0)

X_adj = sm.add_constant(_reg_df[["agreement_10pct", "age_at_admission",
                                  "sex_male", "sofa_total_0_24"]])
mod2 = sm.Logit(_reg_df["death"], X_adj).fit(disp=0)

X_unadj_good = sm.add_constant(_reg_df[["agreement_10pct"]])
mod_good_unadj = sm.Logit(_reg_df["good_outcome"], X_unadj_good).fit(disp=0)
mod_good_adj = sm.Logit(_reg_df["good_outcome"], X_adj).fit(disp=0)

def _or_ci(model, var):
    coef = model.params[var]
    se   = model.bse[var]
    pval = model.pvalues[var]
    return np.exp(coef), np.exp(coef - 1.96*se), np.exp(coef + 1.96*se), pval

print(f"\nMortality (death = CPC5) — OR per 0.1 increase in concordance:")
or_, lo, hi, p = _or_ci(mod1, "agreement_10pct")
print(f"  Unadjusted              : OR = {or_:.3f}  (95% CI {lo:.3f}–{hi:.3f}, p = {p:.4f})")
or_, lo, hi, p = _or_ci(mod2, "agreement_10pct")
print(f"  Adjusted (age/sex/SOFA) : OR = {or_:.3f}  (95% CI {lo:.3f}–{hi:.3f}, p = {p:.4f})")

print(f"\nGood outcome (CPC1_2) — OR per 0.1 increase in concordance:")
or_, lo, hi, p = _or_ci(mod_good_unadj, "agreement_10pct")
print(f"  Unadjusted              : OR = {or_:.3f}  (95% CI {lo:.3f}–{hi:.3f}, p = {p:.4f})")
or_, lo, hi, p = _or_ci(mod_good_adj, "agreement_10pct")
print(f"  Adjusted (age/sex/SOFA) : OR = {or_:.3f}  (95% CI {lo:.3f}–{hi:.3f}, p = {p:.4f})")

print(f"\nFull covariate effects (mortality model):")
print(f"  {'Variable':<25} {'OR':>8} {'95% CI':>20} {'p-value':>10}")
for var in ["agreement_10pct", "age_at_admission", "sex_male", "sofa_total_0_24"]:
    or_, lo, hi, p = _or_ci(mod2, var)
    print(f"  {var:<25} {or_:>8.3f}   {lo:>5.3f}–{hi:.3f}      {p:>9.4f}")

# ── Save ──
per_patient.to_parquet(OUT_DIR / "concordance_eval.parquet", index=False)
print(f"\nSaved per-patient concordance → {OUT_DIR / 'concordance_eval.parquet'}")

EVALUATION — concordance + adjusted outcome regression

Decision-point concordance (val, n=6,098):
  Exact agreement     : 88.1%
  Within 1 tier       : 99.7%
  Mean action distance: 0.12 tiers

Action distribution — clinician vs policy:
  Tier        Clinician     Policy        Δ
  0 Off          14.1%      13.3%    -0.8%
  1 Low          17.3%      15.1%    -2.2%
  2 Med          31.7%      36.7%    +5.0%
  3 High         22.8%      21.5%    -1.3%
  4 VHigh        14.0%      13.3%    -0.7%

Confusion matrix (rows=clinician, cols=policy, % of decisions):
          P:Off   P:Low   P:Med   P:High  P:VHi   
  C:Off      13.3    0.5    0.3    0.0    0.0
  C:Low       0.0   12.8    4.5    0.0    0.0
  C:Med       0.0    1.8   29.5    0.5    0.0
  C:High      0.0    0.0    2.4   19.8    0.6
  C:VHi       0.0    0.0    0.0    1.3   12.7

Per-patient concordance distribution (val, n=140):
count    140.000
mean       0.858
std        0.172
min        0.000
25%        0.822
50%        0.913
75%